In [7]:
import numpy as np
import pylablib as pll

pll.par["devices/dlls/andor_sdk2"] = r"C:\Program Files\Andor SOLIS"
pll.par["devices/dlls/andor_shamrock"] = r"C:\Program Files\Andor SOLIS"

from pylablib.devices import Andor

cam = None
spec = None

cam = Andor.AndorSDK2Camera()
spec = Andor.ShamrockSpectrograph()

print("Camera:", cam.get_device_info())
print("Detector:", cam.get_detector_size())
print("Pixel size:", cam.get_pixel_size())
print("Grating:", spec.get_grating())

cam.set_trigger_mode("int")
cam.set_read_mode("fvb")
cam.set_acquisition_mode("single")
cam.set_exposure(0.05)
cam.set_fan_mode("full")

# Configure the Kymera calibration for this Newton detector.
spec.set_wavelength(633e-9)
print(spec.get_wavelength())
spec.setup_pixels_from_camera(cam)

print("Configured pixels:", spec.get_number_pixels())
print("Configured pixel width:", spec.get_pixel_width())
print(cam.get_fan_mode())
print(cam.get_temperature())
cam.set_cooler(True)
cam.set_temperature(-80)
print(cam.is_cooler_on())
print(cam.get_temperature())

wavelength_nm = np.asarray(
    spec.get_calibration(),
    dtype=float,
) * 1e9

image = np.asarray(cam.snap())
spectrum = np.asarray(image, dtype=float).squeeze()

if spectrum.ndim != 1:
    spectrum = spectrum.sum(axis=0)

spectrum = spectrum.ravel()

print("Image shape:", image.shape)
print("Spectrum shape:", spectrum.shape)
print("Calibration shape:", wavelength_nm.shape)

if len(spectrum) != len(wavelength_nm):
    raise RuntimeError(
        f"Length mismatch: spectrum={len(spectrum)}, "
        f"calibration={len(wavelength_nm)}"
    )

print(
    "Wavelength range:",
    wavelength_nm[0],
    "to",
    wavelength_nm[-1],
    "nm",
)

Camera: TDeviceInfo(controller_model='USB', head_model='DU920P_BVF', serial_number=22996)
Detector: (1024, 255)
Pixel size: (2.6e-05, 2.6e-05)
Grating: 2
6.33e-07
Configured pixels: 1024
Configured pixel width: 2.6e-05
full
-80.54499816894531
True
-80.54499816894531
Image shape: (1, 1024)
Spectrum shape: (1024,)
Calibration shape: (1024,)
Wavelength range: 604.68212890625 to 660.685791015625 nm


In [2]:
import os
import psutil

for module in psutil.Process(os.getpid()).memory_maps():
    path = module.path
    if any(x in path.lower() for x in [
        "atmcd",
        "shamrock",
        "atspectrograph"
    ]):
        print(path)

C:\Program Files\Andor SOLIS\atmcd64d_legacy.dll
C:\Program Files\Andor SOLIS\atspectrograph.dll
C:\Program Files\Andor SOLIS\atmcd64d.dll


In [2]:
spec.set_grating(2)
print(spec.get_grating())

2
